In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../'))
sys.path.append(base_path)

import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from tqdm import tqdm
import pandas as pd
from IPython.display import display
import shutil

import project_images_loader as pil
import utils.util as util
import utils.functions as fc
import utils.nifti_functions as nfc
import utils.visualization as vis
from scipy import stats
from PIL import Image
np.set_printoptions(precision=4)
from PIL.ExifTags import TAGS
import test_code.creator_2D_images.constants as cts_prep_mni

val_ids = ["08202"] + pil.get_matching_ids()[621:]


SERVER: False


# Copy karl results

In [2]:
# # ct_path = "/cluster/work/karlehof/hunt_mri/data/ct/"
# karl_results_path = f'{current_path}/images/karl_results/'
# archivos = os.listdir(karl_results_path)
# archivos = [archivo for archivo in archivos if os.path.isfile(os.path.join(karl_results_path, archivo))]
# ids = [int(archivo.split(".")[0]) for archivo in archivos]
# ids.sort()


# # print(ids[0])
# for id in ids:
#     image = Image.open(f"/cluster/work/karlehof/hunt_mri/data/ct/{id}.png")
#     location = image.text["OrigLocation"]
#     nombre_sin_extension = os.path.splitext(os.path.basename(location))[0]
#     split_name = nombre_sin_extension.split("_")
#     real_id = "{:05}".format(int(split_name[0]))

#     origin_folder = f"{current_path}/images/karl_results"
#     target_folder = f"{current_path}/images/results/{real_id}"
#     if fc.verify_folder(target_folder, create=True):
#         shutil.copy2(f"{origin_folder}/{id}.png", f"{target_folder}/{nombre_sin_extension}.png")



# Reconstrcut 3D image

In [3]:
from skimage.exposure import match_histograms
results_2D = f"{current_path}/images/results_2D/"
VAL_IDS_2 = [nombre for nombre in os.listdir(results_2D) if os.path.isdir(os.path.join(results_2D, nombre))]
print(len(VAL_IDS_2))

def create_3D_image(id):
    img_hunt3, affine = nfc.load_nifti(cts_prep_mni.get_registered_image_path_name(3, id))
    img_res = np.zeros_like(img_hunt3, dtype=np.float32)

    for layer in range(pil.LAYER_START_2D, pil.LAYER_END_2D):
        image = Image.open(f"{results_2D}/{id}/{id}_1_T1_{layer}.png")
        image = np.array(image, dtype=np.float32) / 255

        img_res[16:176, 3:227, layer] = image
    return img_res, affine

def save_3D_image():
    for id in VAL_IDS_2:
        img_res, affine = create_3D_image(id)
        img_path = f"{current_path}/images/results/{id}/{id}_0_T1_RES.nii.gz"
        if fc.verify_folder(os.path.dirname(img_path), create=True):
            nfc.save_nifti(img_res, affine, img_path)
        
# save_3D_image()
# id = "08202"
# img_res, affine = create_3D_image(id)
# img_hunt3, affine = nfc.load_nifti(cts_prep_mni.get_registered_image_path_name(4, id))
# diff = np.abs(img_res - img_hunt3)
# %matplotlib widget

# # fc.imgshow_3D_list(id, [img_res, img_hunt3, diff], subimg_titles=["img_res", "img_hunt3", "img_diff"])
# fc.imgshow_3D_list(id, [img_res, img_hunt3], subimg_titles=["img_res", "img_hunt3"])
# # img = Image.open('whatever.png')
# # # Create a new image of the original size with black background
# # result = Image.new(img.mode, (229,193), (0, 0, 0))
# # # Pase the cropped image at the offset position (same as I cropped it)
# # result.paste(img, (3, 16))
# # result.save('whatever_fixed.png')

140


# selected ids

In [17]:
results_2D = f"{current_path}/images/results_postprocessed_all/"
VAL_IDS_2 = [nombre for nombre in os.listdir(results_2D) if os.path.isdir(os.path.join(results_2D, nombre))]
VAL_IDS_2.sort()

val_images = ["08202"] + pil.get_matching_ids()[621:]

repetidos = list(set(VAL_IDS_2) & set(val_images))
repetidos.sort()
VAL_IDS_2_sin_repetidos = [elemento for elemento in VAL_IDS_2 if elemento not in repetidos]
VAL_IDS_2_sin_repetidos.sort()
final_list = repetidos + VAL_IDS_2_sin_repetidos[:83 - len(repetidos)]
final_list.sort()

# for id in final_list:
#     res_post_all = f"{current_path}/images/results_postprocessed_all/{id}"
#     res_post = f"{current_path}/images/results_postprocessed/{id}"
#     if fc.verify_folder(res_post, create=True):
#         # print(f"{res_post_all}/{id}_0_T1_REG.nii.gz", f"{res_post}/{id}_0_T1_REG.nii.gz")
#         shutil.copy2(f"{res_post_all}/{id}_0_T1_REG_RES.nii.gz", f"{res_post}/{id}_0_T1_REG_RES.nii.gz")

# print(VAL_IDS_2_sin_repetidos)
# print(len(VAL_IDS_2_sin_repetidos))
# print(repetidos)
# print(len(repetidos))
# print(final_list)
# print(len(final_list))